# Knee MRI v1 -- Shared Backbone, Per-Plane Pooling, Joint Multi-Label Head

Baseline architecture for the RSNA Knee Abnormality Detection competition.

**Design** (matches the real community baseline
[JunhaoLiXD/RSNA_Knee_Abnormality_Detection](https://github.com/JunhaoLiXD/RSNA_Knee_Abnormality_Detection)
and the published [CoPAS](https://www.nature.com/articles/s41467-024-51888-4) architecture --
not a per-plane-submodel design):

```
For each of the 3 planes (Sagittal / Coronal / Axial):
    pick 1 priority series (fluid-sensitive fat-sat preferred)
    -> sample/pad to N_SLICES slices
    -> shared EfficientNet-B0 backbone (same weights for all 3 planes)
    -> attention-pool over slices -> 1 embedding per plane
concatenate the 3 plane embeddings
    -> shared MLP head -> 12 sigmoid outputs (one joint model, not 3 submodels)
```

## Why this version looks different from the first draft

The first draft ran the raw-image pipeline end-to-end every epoch and measured 441s/epoch
locally (RTX 4050) and 600s/epoch on Kaggle (2x T4) -- both with GPU utilization sitting at
40-50% and most of VRAM free, which means it was never GPU-bound. Two real problems, found
from the actual profiler screenshots, not guessed:

1. **The backbone is frozen** (`FREEZE_BACKBONE=True`) -- its output for a given slice never
   changes -- yet the raw pipeline recomputed it from scratch on every epoch. 15 epochs of
   fully redundant CNN forward passes on data that never changes.
2. **`.npz` files are zlib-compressed** (~726KB on disk vs. ~1.15MB raw) and were being
   decompressed synchronously, one sample at a time, in the main process (`num_workers=0`),
   serialized with GPU compute -- so the GPU sat idle waiting for data instead of computing.

Fix: **precompute the frozen backbone's per-slice features once**, parallelized across CPU
threads for the decompression-heavy load step, cache them to disk, and train the (tiny,
actually-trainable) attention-pool + head on those cached vectors. That turns "15x redundant
CNN passes over compressed images" into "1x CNN pass (parallel-loaded) + 15x cheap MLP
passes over small cached vectors".

**Note on `src/model/*.py`**: this repo already has a more complex DINOv2 + Region-Guided-
Attention implementation there (from an earlier planning pass), untouched by this notebook.
It's a reasonable v2 direction once this baseline is validated, but has stale paths and no
gold-label upweighting.

Run this locally (`torch_gpu` conda env). Widen `FOLDS_TO_RUN` to all 5 and point the data
paths at the Kaggle dataset mounts for the real submission run -- the hidden test images
aren't available locally, only the pipeline can be exercised end-to-end here.

# Model Architecture:
<img src = "../../docs/images/model_architecture_v1.png">

In [33]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import roc_auc_score
from concurrent.futures import ThreadPoolExecutor

TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture',
]
PLANES = ['Sagittal', 'Coronal', 'Axial']

In [34]:
# -----------------------------------------------------------------------
# Config -- swap DATA_DIR for the Kaggle dataset mount path when running there
# -----------------------------------------------------------------------
DATA_DIR = 'D:/Coding/kaggle/RSNA/data'
MAIN_DIR = f'{DATA_DIR}/main_data'
NPZ_DIR = f'{DATA_DIR}/data'

CONSENSUS_LABELS_CSV = f'{MAIN_DIR}/final/train_folds_consensus.csv'
SERIES_META_CSV = f'{MAIN_DIR}/train_series.csv'
GOLD_LABELS_CSV = f'{MAIN_DIR}/to_use/available_labels.csv'
TEST_CSV = f'{MAIN_DIR}/test.csv'
TEST_SERIES_CSV = f'{MAIN_DIR}/test_series.csv'
SAMPLE_SUB_CSV = f'{MAIN_DIR}/sample_submission.csv'

OUTPUT_DIR = 'D:/Coding/kaggle/RSNA/output/v1'
FEATURE_CACHE_DIR = f'{OUTPUT_DIR}/feature_cache'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)

# Data / model shape
N_SLICES = 12            # slices sampled per series (originals are variable length, ~24)
IMG_SIZE = 224
FREEZE_BACKBONE = True   # Phase 1: frozen backbone, train pooling + head only

# Precompute (one-time, CPU-heavy: file read + zlib decompress + backbone forward)
PRECOMPUTE_BATCH_SIZE = 16
NUM_LOAD_WORKERS = max(1, (os.cpu_count() or 2) - 1)   # leave 1 core free for the OS/notebook

# Training (fast, on cached feature vectors -- no CNN in this loop, workers not needed)
TRAIN_BATCH_SIZE = 32
TRAIN_NUM_WORKERS = 0
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
GOLD_WEIGHT = 8.0         # upweight the 58 genuine gold rows, matches the real community baseline
USE_AMP = torch.cuda.is_available()

# Which folds to actually train in this run.
# [0] = fast sanity run (~1/5 of the data). Use list(range(5)) for the real, full CV run.
FOLDS_TO_RUN = list(range(5))

SEED = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | loader threads: {NUM_LOAD_WORKERS} (of {os.cpu_count()} logical cores)')

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

Device: cuda | loader threads: 11 (of 12 logical cores)


## 1. Load labels, series metadata, and pick one priority series per plane

In [35]:
consensus_df = pd.read_csv(CONSENSUS_LABELS_CSV)
series_df = pd.read_csv(SERIES_META_CSV)
gold_df = pd.read_csv(GOLD_LABELS_CSV)

gold_ids = set(gold_df['StudyInstanceUID'])
print(f'Studies: {len(consensus_df)} | series: {len(series_df)} | gold rows: {len(gold_ids)}')


def build_priority_series(series_meta):
    """One series per (study, plane): prefer Fluid_Sensitive==1 (fat-sat), else any."""
    priority = {}
    series_meta = series_meta.sort_values('SeriesInstanceUID')
    for (uid, plane), grp in series_meta.groupby(['StudyInstanceUID', 'Anatomical_Plane']):
        fluid = grp[grp['Fluid_Sensitive'] == 1]
        chosen = fluid.iloc[0] if len(fluid) else grp.iloc[0]
        priority[(uid, plane)] = chosen['SeriesInstanceUID']
    return priority


train_priority_series = build_priority_series(series_df)
print(f'Priority series assigned: {len(train_priority_series)} '
      f"(expect studies x 3 planes = {consensus_df['StudyInstanceUID'].nunique() * 3})")

Studies: 4407 | series: 24371 | gold rows: 58
Priority series assigned: 13221 (expect studies x 3 planes = 13221)


## 2. Raw-image dataset & model

Kept for reference and for a later Phase-2 (unfreeze backbone, fine-tune end-to-end). v1
does **not** train through this path directly -- see Section 3, which precomputes this
backbone's output once and trains only the lightweight pooling + head on the cached result.

In [36]:
class KneeMultiPlaneDataset(Dataset):
    """One sample = one study. Loads its 3 priority series (one per plane), raw + augmented."""

    def __init__(self, study_ids, priority_series, npz_dir, n_slices, img_size,
                 labels_df=None, gold_ids=None, gold_weight=1.0, is_train=False):
        self.study_ids = list(study_ids)
        self.priority_series = priority_series
        self.npz_dir = npz_dir
        self.n_slices = n_slices
        self.img_size = img_size
        self.is_train = is_train
        self.gold_ids = gold_ids or set()
        self.gold_weight = gold_weight

        self.label_lookup = None
        if labels_df is not None:
            idx = labels_df.set_index('StudyInstanceUID')[TARGET_COLS]
            self.label_lookup = {uid: idx.loc[uid].astype(float).values for uid in self.study_ids}

    def __len__(self):
        return len(self.study_ids)

    def _load_plane(self, uid, plane):
        series_uid = self.priority_series.get((uid, plane))
        if series_uid is None:
            return np.zeros((self.n_slices, self.img_size, self.img_size), dtype=np.float32)
        path = os.path.join(self.npz_dir, f'{series_uid}.npz')
        try:
            vol = np.load(path)['data'].astype(np.float32) / 255.0   # (T, H, W)
        except Exception:
            return np.zeros((self.n_slices, self.img_size, self.img_size), dtype=np.float32)

        T = vol.shape[0]
        if T == self.n_slices:
            out = vol
        elif T > self.n_slices:
            idx = np.linspace(0, T - 1, self.n_slices).astype(int)
            out = vol[idx]
        else:
            pad = np.zeros((self.n_slices - T, self.img_size, self.img_size), dtype=np.float32)
            out = np.concatenate([vol, pad], axis=0)

        if self.is_train and np.random.rand() < 0.5:
            out = out[:, :, ::-1].copy()
        return out

    def __getitem__(self, idx):
        uid = self.study_ids[idx]
        item = {'study_uid': uid}
        for plane, key in zip(PLANES, ['sagittal', 'coronal', 'axial']):
            arr = self._load_plane(uid, plane)                               # (T, H, W)
            t = torch.from_numpy(arr).unsqueeze(1).repeat(1, 3, 1, 1)         # (T, 3, H, W)
            t = (t - IMAGENET_MEAN) / IMAGENET_STD
            item[key] = t

        if self.label_lookup is not None:
            item['labels'] = torch.tensor(self.label_lookup[uid], dtype=torch.float32)
            item['weight'] = torch.tensor(self.gold_weight if uid in self.gold_ids else 1.0, dtype=torch.float32)
        return item

In [37]:
class SliceBackbone(nn.Module):
    def __init__(self, freeze=True):
        super().__init__()
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = m.features
        self.pool = m.avgpool
        self.out_dim = 1280
        if freeze:
            for p in self.features.parameters():
                p.requires_grad = False

    def forward(self, x):        # x: (N, 3, H, W)
        f = self.features(x)
        return self.pool(f).flatten(1)   # (N, out_dim)


class AttentionPool(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x):        # x: (B, T, dim)
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1)   # (B, dim)


class KneeMultiPlaneModel(nn.Module):
    """Raw end-to-end model (backbone + pooling + head). Used for Phase-2 fine-tuning only."""

    def __init__(self, n_targets=12, freeze_backbone=True):
        super().__init__()
        self.backbone = SliceBackbone(freeze=freeze_backbone)
        self.slice_pool = AttentionPool(self.backbone.out_dim)
        fused_dim = self.backbone.out_dim * 3
        self.head = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, n_targets),
        )

    def encode_plane(self, x):   # x: (B, T, 3, H, W)
        B, T, C, H, W = x.shape
        feat = self.backbone(x.view(B * T, C, H, W))
        feat = feat.view(B, T, -1)
        return self.slice_pool(feat)     # (B, dim)

    def forward(self, sagittal, coronal, axial):
        s = self.encode_plane(sagittal)
        c = self.encode_plane(coronal)
        a = self.encode_plane(axial)
        fused = torch.cat([s, c, a], dim=1)
        return self.head(fused)          # (B, n_targets) raw logits

## 3. Precompute frozen-backbone features once -- the actual speed fix

`_load_study_planes` does the CPU-heavy part (file read + zlib decompress + slice
sampling) and is parallelized across `NUM_LOAD_WORKERS` threads -- threads, not processes,
because Windows/Jupyter cannot pickle notebook-defined classes/functions into spawned worker
processes (the well-known `DataLoader(num_workers>0)`-in-a-Windows-notebook failure mode).
Threads sidestep that entirely (shared memory, no pickling), and both file I/O and zlib
decompression release the GIL, so real parallelism is still achieved for exactly the part
that was serialized before.

The GPU-bound part (the backbone forward pass) stays sequential on the main thread -- there's
only one GPU, so nothing is gained by trying to parallelize that part too.

This runs once per study (cached to disk, `skip_existing=True`), regardless of how many
epochs or folds you train afterward.

In [38]:
def _load_study_planes(uid, priority_series):
    """CPU-bound: read + decompress the 3 priority-series npz files for one study."""
    out = {}
    for plane, key in zip(PLANES, ['sagittal', 'coronal', 'axial']):
        series_uid = priority_series.get((uid, plane))
        arr = None
        if series_uid is not None:
            path = os.path.join(NPZ_DIR, f'{series_uid}.npz')
            try:
                vol = np.load(path)['data'].astype(np.float32) / 255.0   # (T, H, W)
                T = vol.shape[0]
                if T == N_SLICES:
                    arr = vol
                elif T > N_SLICES:
                    idx = np.linspace(0, T - 1, N_SLICES).astype(int)
                    arr = vol[idx]
                else:
                    pad = np.zeros((N_SLICES - T, IMG_SIZE, IMG_SIZE), dtype=np.float32)
                    arr = np.concatenate([vol, pad], axis=0)
            except Exception:
                arr = None
        if arr is None:
            arr = np.zeros((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        out[key] = arr
    return uid, out


@torch.no_grad()
def precompute_features(study_ids, priority_series, backbone, cache_dir,
                         batch_size=PRECOMPUTE_BATCH_SIZE, num_load_workers=NUM_LOAD_WORKERS,
                         skip_existing=True):
    todo = [u for u in study_ids if not (skip_existing and os.path.exists(os.path.join(cache_dir, f'{u}.npz')))]
    print(f'{len(todo)} / {len(study_ids)} studies need feature extraction '
          f'({num_load_workers} loader threads, batch size {batch_size})')
    if not todo:
        return

    backbone.eval()
    t0 = time.time()
    done = 0

    with ThreadPoolExecutor(max_workers=num_load_workers) as pool:
        for start in range(0, len(todo), batch_size):
            batch_ids = todo[start: start + batch_size]
            results = list(pool.map(lambda u: _load_study_planes(u, priority_series), batch_ids))

            feats = {}
            for key in ['sagittal', 'coronal', 'axial']:
                raw = np.stack([r[1][key] for r in results])                  # (B, T, H, W)
                x = torch.from_numpy(raw).unsqueeze(2).repeat(1, 1, 3, 1, 1)   # (B, T, 3, H, W)
                x = (x - IMAGENET_MEAN) / IMAGENET_STD
                x = x.to(device)
                B, T, C, H, W = x.shape
                feats[key] = backbone(x.view(B * T, C, H, W)).view(B, T, -1).cpu().numpy()

            for b, (uid, _) in enumerate(results):
                np.savez(os.path.join(cache_dir, f'{uid}.npz'),
                         sagittal=feats['sagittal'][b], coronal=feats['coronal'][b], axial=feats['axial'][b])

            done += len(batch_ids)
            if (start // batch_size) % 20 == 0 or done == len(todo):
                elapsed = time.time() - t0
                rate = done / elapsed if elapsed > 0 else 0.0
                print(f'  {done}/{len(todo)} studies | {elapsed:.0f}s elapsed | {rate:.2f} studies/s')

    print(f'Precompute finished: {done} studies in {time.time() - t0:.0f}s -> {cache_dir}')

In [39]:
class KneeFeatureDataset(Dataset):
    """Loads cached (T, 1280) per-plane backbone features for one study -- no CNN, no decompression."""

    def __init__(self, study_ids, cache_dir, labels_df=None, gold_ids=None, gold_weight=1.0):
        self.study_ids = list(study_ids)
        self.cache_dir = cache_dir
        self.gold_ids = gold_ids or set()
        self.gold_weight = gold_weight
        self.label_lookup = None
        if labels_df is not None:
            idx = labels_df.set_index('StudyInstanceUID')[TARGET_COLS]
            self.label_lookup = {uid: idx.loc[uid].astype(float).values for uid in self.study_ids}

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        uid = self.study_ids[idx]
        d = np.load(os.path.join(self.cache_dir, f'{uid}.npz'))
        item = {
            'study_uid': uid,
            'sagittal': torch.from_numpy(d['sagittal']).float(),
            'coronal': torch.from_numpy(d['coronal']).float(),
            'axial': torch.from_numpy(d['axial']).float(),
        }
        if self.label_lookup is not None:
            item['labels'] = torch.tensor(self.label_lookup[uid], dtype=torch.float32)
            item['weight'] = torch.tensor(self.gold_weight if uid in self.gold_ids else 1.0, dtype=torch.float32)
        return item


class KneeFeatureModel(nn.Module):
    """Trains only the attention-pool + head on precomputed, frozen backbone features."""

    def __init__(self, feat_dim=1280, n_targets=12):
        super().__init__()
        self.slice_pool = AttentionPool(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, n_targets),
        )

    def forward(self, sagittal, coronal, axial):
        s = self.slice_pool(sagittal)
        c = self.slice_pool(coronal)
        a = self.slice_pool(axial)
        fused = torch.cat([s, c, a], dim=1)
        return self.head(fused)          # (B, n_targets) raw logits

In [40]:
# Run once: precomputes + caches features for every study in the labeled set.
# Safe to re-run -- skip_existing=True means already-cached studies are skipped instantly.
backbone = SliceBackbone(freeze=FREEZE_BACKBONE).to(device)
precompute_features(consensus_df['StudyInstanceUID'].tolist(), train_priority_series, backbone, FEATURE_CACHE_DIR)

0 / 4407 studies need feature extraction (11 loader threads, batch size 16)


## 4. Loss & metrics

In [41]:
def weighted_bce(logits, targets, row_weight, pos_weight):
    per_elem = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none')
    per_sample = per_elem.mean(dim=1)
    return (per_sample * row_weight).sum() / row_weight.sum()


def macro_auc(y_true_bin, y_prob, cols):
    """y_true_bin: (N, len(cols)) hard 0/1 labels. Skips columns with only one class present."""
    per_class = {}
    for i, c in enumerate(cols):
        yt = y_true_bin[:, i]
        if len(np.unique(yt)) < 2:
            continue
        per_class[c] = roc_auc_score(yt, y_prob[:, i])
    macro = float(np.mean(list(per_class.values()))) if per_class else float('nan')
    return macro, per_class

## 5. Train / eval loop functions

In [42]:
def train_one_epoch(model, loader, optimizer, pos_weight, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        sag, cor, ax = batch['sagittal'].to(device), batch['coronal'].to(device), batch['axial'].to(device)
        y, w = batch['labels'].to(device), batch['weight'].to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(sag, cor, ax)
            loss = weighted_bce(logits, y, w, pos_weight)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * sag.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs, all_uids = [], []
    for batch in loader:
        sag, cor, ax = batch['sagittal'].to(device), batch['coronal'].to(device), batch['axial'].to(device)
        logits = model(sag, cor, ax)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_uids.extend(batch['study_uid'])
    return np.concatenate(all_probs, axis=0), all_uids

## 6. Run training (per fold)

In [43]:
consensus_indexed = consensus_df.set_index('StudyInstanceUID')
gold_indexed = gold_df.set_index('StudyInstanceUID')

oof_probs = {}
fold_ckpts = {}

for fold in FOLDS_TO_RUN:
    print(f"\n{'=' * 72}\nFold {fold}\n{'=' * 72}")
    train_ids = consensus_df.loc[consensus_df.fold != fold, 'StudyInstanceUID'].tolist()
    val_ids = consensus_df.loc[consensus_df.fold == fold, 'StudyInstanceUID'].tolist()
    print(f'train studies: {len(train_ids)} | val studies: {len(val_ids)}')

    train_ds = KneeFeatureDataset(train_ids, FEATURE_CACHE_DIR,
                                   labels_df=consensus_df, gold_ids=gold_ids, gold_weight=GOLD_WEIGHT)
    val_ds = KneeFeatureDataset(val_ids, FEATURE_CACHE_DIR,
                                 labels_df=consensus_df, gold_ids=gold_ids, gold_weight=GOLD_WEIGHT)

    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                               num_workers=TRAIN_NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
                             num_workers=TRAIN_NUM_WORKERS, pin_memory=True)

    prevalence = consensus_df.loc[consensus_df.fold != fold, TARGET_COLS].ge(0.5).mean().values
    pos_weight = torch.tensor(np.clip((1 - prevalence) / np.clip(prevalence, 1e-3, None), 1.0, 10.0),
                               dtype=torch.float32).to(device)

    model = KneeFeatureModel(feat_dim=1280, n_targets=len(TARGET_COLS)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_gold_auc = -1.0
    ckpt_path = os.path.join(OUTPUT_DIR, f'v1_fold{fold}.pt')

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, pos_weight, scaler)
        val_probs, val_uids = predict(model, val_loader)

        val_soft_true = consensus_indexed.loc[val_uids, TARGET_COLS].values
        noisy_auc, _ = macro_auc((val_soft_true >= 0.5).astype(int), val_probs, TARGET_COLS)

        gold_mask = np.array([u in gold_ids for u in val_uids])
        if gold_mask.any():
            gold_uids_here = [u for u, m in zip(val_uids, gold_mask) if m]
            gold_true = gold_indexed.loc[gold_uids_here, TARGET_COLS].values.astype(int)
            gold_auc, _ = macro_auc(gold_true, val_probs[gold_mask], TARGET_COLS)
        else:
            gold_auc = float('nan')

        marker = ''
        if not np.isnan(gold_auc) and gold_auc > best_gold_auc:
            best_gold_auc = gold_auc
            marker = ' *'
            torch.save(model.state_dict(), ckpt_path)

        print(f'epoch {epoch:>2}/{EPOCHS} | loss {train_loss:.4f} | '
              f'noisy AUC {noisy_auc:.4f} | gold AUC {gold_auc:.4f} | '
              f'{time.time() - t0:>5.1f}s{marker}')

    if not os.path.exists(ckpt_path):     # fallback: fold had no gold rows to select on
        torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    val_probs, val_uids = predict(model, val_loader)
    for u, p in zip(val_uids, val_probs):
        oof_probs[u] = p
    fold_ckpts[fold] = ckpt_path
    print(f'Fold {fold} best gold AUC: {best_gold_auc:.4f} -> {ckpt_path}')


Fold 0
train studies: 3525 | val studies: 882


C:\Users\nishk\AppData\Local\Temp\ipykernel_59164\3540300145.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch  1/20 | loss 1.0075 | noisy AUC 0.7389 | gold AUC 0.6562 |   4.7s *
epoch  2/20 | loss 0.9571 | noisy AUC 0.7600 | gold AUC 0.6962 |   4.4s *
epoch  3/20 | loss 0.9300 | noisy AUC 0.7709 | gold AUC 0.7238 |   4.1s *
epoch  4/20 | loss 0.9145 | noisy AUC 0.7760 | gold AUC 0.7753 |   4.1s *
epoch  5/20 | loss 0.8995 | noisy AUC 0.7805 | gold AUC 0.7928 |   4.0s *
epoch  6/20 | loss 0.8820 | noisy AUC 0.7776 | gold AUC 0.7966 |   4.0s *
epoch  7/20 | loss 0.8683 | noisy AUC 0.7794 | gold AUC 0.7835 |   4.0s
epoch  8/20 | loss 0.8542 | noisy AUC 0.7818 | gold AUC 0.7983 |   4.0s *
epoch  9/20 | loss 0.8458 | noisy AUC 0.7752 | gold AUC 0.7808 |   4.1s
epoch 10/20 | loss 0.8323 | noisy AUC 0.7787 | gold AUC 0.8030 |   4.1s *
epoch 11/20 | loss 0.8224 | noisy AUC 0.7763 | gold AUC 0.7848 |   4.1s
epoch 12/20 | loss 0.8120 | noisy AUC 0.7832 | gold AUC 0.7988 |   4.1s
epoch 13/20 | loss 0.7963 | noisy AUC 0.7774 | gold AUC 0.7659 |   4.0s
epoch 14/20 | loss 0.7891 | noisy AUC 0.7738 | g

C:\Users\nishk\AppData\Local\Temp\ipykernel_59164\3540300145.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch  1/20 | loss 0.9982 | noisy AUC 0.7217 | gold AUC 0.7746 |   4.1s *
epoch  2/20 | loss 0.9494 | noisy AUC 0.7370 | gold AUC 0.7686 |   4.0s
epoch  3/20 | loss 0.9212 | noisy AUC 0.7432 | gold AUC 0.8339 |   4.0s *
epoch  4/20 | loss 0.9040 | noisy AUC 0.7516 | gold AUC 0.8099 |   4.0s
epoch  5/20 | loss 0.8882 | noisy AUC 0.7572 | gold AUC 0.8325 |   4.1s
epoch  6/20 | loss 0.8712 | noisy AUC 0.7567 | gold AUC 0.8235 |   4.1s
epoch  7/20 | loss 0.8556 | noisy AUC 0.7585 | gold AUC 0.8135 |   4.1s
epoch  8/20 | loss 0.8471 | noisy AUC 0.7599 | gold AUC 0.8027 |   4.1s
epoch  9/20 | loss 0.8373 | noisy AUC 0.7573 | gold AUC 0.8276 |   4.1s
epoch 10/20 | loss 0.8196 | noisy AUC 0.7598 | gold AUC 0.8040 |   4.0s
epoch 11/20 | loss 0.8097 | noisy AUC 0.7590 | gold AUC 0.8221 |   4.1s
epoch 12/20 | loss 0.7998 | noisy AUC 0.7558 | gold AUC 0.7948 |   4.0s
epoch 13/20 | loss 0.7891 | noisy AUC 0.7488 | gold AUC 0.8409 |   4.1s *
epoch 14/20 | loss 0.7842 | noisy AUC 0.7526 | gold AUC 0.

C:\Users\nishk\AppData\Local\Temp\ipykernel_59164\3540300145.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch  1/20 | loss 1.0014 | noisy AUC 0.7163 | gold AUC 0.7225 |   4.1s *
epoch  2/20 | loss 0.9579 | noisy AUC 0.7418 | gold AUC 0.6452 |   4.0s
epoch  3/20 | loss 0.9310 | noisy AUC 0.7450 | gold AUC 0.6907 |   4.1s
epoch  4/20 | loss 0.9112 | noisy AUC 0.7575 | gold AUC 0.7111 |   4.0s
epoch  5/20 | loss 0.8906 | noisy AUC 0.7604 | gold AUC 0.7303 |   4.0s *
epoch  6/20 | loss 0.8789 | noisy AUC 0.7636 | gold AUC 0.7371 |   4.1s *
epoch  7/20 | loss 0.8647 | noisy AUC 0.7586 | gold AUC 0.7295 |   4.0s
epoch  8/20 | loss 0.8504 | noisy AUC 0.7660 | gold AUC 0.7351 |   4.3s
epoch  9/20 | loss 0.8336 | noisy AUC 0.7653 | gold AUC 0.7250 |   4.0s
epoch 10/20 | loss 0.8235 | noisy AUC 0.7670 | gold AUC 0.7258 |   4.0s
epoch 11/20 | loss 0.8122 | noisy AUC 0.7675 | gold AUC 0.7384 |   4.1s *
epoch 12/20 | loss 0.8070 | noisy AUC 0.7570 | gold AUC 0.6922 |   4.1s
epoch 13/20 | loss 0.7922 | noisy AUC 0.7581 | gold AUC 0.7068 |   4.0s
epoch 14/20 | loss 0.7782 | noisy AUC 0.7603 | gold AUC 

C:\Users\nishk\AppData\Local\Temp\ipykernel_59164\3540300145.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch  1/20 | loss 1.0063 | noisy AUC 0.7218 | gold AUC 0.6468 |   4.1s *
epoch  2/20 | loss 0.9549 | noisy AUC 0.7360 | gold AUC 0.6445 |   4.1s
epoch  3/20 | loss 0.9340 | noisy AUC 0.7469 | gold AUC 0.6728 |   4.0s *
epoch  4/20 | loss 0.9174 | noisy AUC 0.7511 | gold AUC 0.6820 |   4.0s *
epoch  5/20 | loss 0.8964 | noisy AUC 0.7530 | gold AUC 0.6946 |   4.0s *
epoch  6/20 | loss 0.8764 | noisy AUC 0.7572 | gold AUC 0.6733 |   4.0s
epoch  7/20 | loss 0.8626 | noisy AUC 0.7579 | gold AUC 0.6855 |   4.1s
epoch  8/20 | loss 0.8488 | noisy AUC 0.7575 | gold AUC 0.6829 |   4.0s
epoch  9/20 | loss 0.8436 | noisy AUC 0.7520 | gold AUC 0.6789 |   4.0s
epoch 10/20 | loss 0.8210 | noisy AUC 0.7544 | gold AUC 0.6521 |   4.2s
epoch 11/20 | loss 0.8138 | noisy AUC 0.7503 | gold AUC 0.6634 |   4.2s
epoch 12/20 | loss 0.7954 | noisy AUC 0.7554 | gold AUC 0.6495 |   4.1s
epoch 13/20 | loss 0.7804 | noisy AUC 0.7550 | gold AUC 0.6496 |   4.1s
epoch 14/20 | loss 0.7791 | noisy AUC 0.7510 | gold AUC 

C:\Users\nishk\AppData\Local\Temp\ipykernel_59164\3540300145.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch  1/20 | loss 1.0034 | noisy AUC 0.7205 | gold AUC 0.6636 |   4.1s *
epoch  2/20 | loss 0.9515 | noisy AUC 0.7476 | gold AUC 0.7089 |   4.0s *
epoch  3/20 | loss 0.9253 | noisy AUC 0.7576 | gold AUC 0.7394 |   4.1s *
epoch  4/20 | loss 0.9067 | noisy AUC 0.7614 | gold AUC 0.7394 |   4.2s
epoch  5/20 | loss 0.8882 | noisy AUC 0.7662 | gold AUC 0.7284 |   4.1s
epoch  6/20 | loss 0.8766 | noisy AUC 0.7679 | gold AUC 0.7418 |   4.3s *
epoch  7/20 | loss 0.8634 | noisy AUC 0.7679 | gold AUC 0.8008 |   4.0s *
epoch  8/20 | loss 0.8519 | noisy AUC 0.7676 | gold AUC 0.7528 |   4.1s
epoch  9/20 | loss 0.8391 | noisy AUC 0.7730 | gold AUC 0.7737 |   4.1s
epoch 10/20 | loss 0.8299 | noisy AUC 0.7742 | gold AUC 0.7657 |   4.0s
epoch 11/20 | loss 0.8094 | noisy AUC 0.7699 | gold AUC 0.7719 |   4.0s
epoch 12/20 | loss 0.7998 | noisy AUC 0.7663 | gold AUC 0.7483 |   4.1s
epoch 13/20 | loss 0.7874 | noisy AUC 0.7673 | gold AUC 0.7568 |   4.0s
epoch 14/20 | loss 0.7764 | noisy AUC 0.7641 | gold AU

## 7. Out-of-fold summary

In [44]:
oof_uids = list(oof_probs.keys())
oof_matrix = np.stack([oof_probs[u] for u in oof_uids])

oof_soft_true = consensus_indexed.loc[oof_uids, TARGET_COLS].values
noisy_macro, noisy_per_class = macro_auc((oof_soft_true >= 0.5).astype(int), oof_matrix, TARGET_COLS)

gold_mask = np.array([u in gold_ids for u in oof_uids])
gold_oof_uids = [u for u, m in zip(oof_uids, gold_mask) if m]
gold_true = gold_indexed.loc[gold_oof_uids, TARGET_COLS].values.astype(int)
gold_macro, gold_per_class = macro_auc(gold_true, oof_matrix[gold_mask], TARGET_COLS)

print(f'OOF studies: {len(oof_uids)} (of which genuinely-labeled: {len(gold_oof_uids)})')
print(f'Noisy macro AUC  (vs. consensus soft pseudo-labels): {noisy_macro:.4f}')
print(f'HONEST macro AUC (vs. the 58 real gold labels):      {gold_macro:.4f}   <-- this is the number that matters')

print('\nPer-class honest AUC:')
for c, v in sorted(gold_per_class.items(), key=lambda x: -x[1]):
    print(f'  {c:<20} {v:.4f}')

OOF studies: 4407 (of which genuinely-labeled: 58)
Noisy macro AUC  (vs. consensus soft pseudo-labels): 0.7553
HONEST macro AUC (vs. the 58 real gold labels):      0.7562   <-- this is the number that matters

Per-class honest AUC:
  Medial OA            0.8760
  Effusion             0.8683
  PF OA                0.8443
  Baker's              0.8424
  Lateral OA           0.7969
  Lateral Meniscus     0.7615
  Medial Meniscus      0.7404
  Contusion            0.7382
  Synovitis            0.7192
  Fracture             0.6972
  MCL                  0.6327
  ACL                  0.5576


## 8. Inference on test set + submission file

The 3 studies in `test.csv` are just a public stub -- their `.npz` files are almost
certainly not present in this local `NPZ_DIR`, so this cell mainly exercises the plumbing
end-to-end. Point `NPZ_DIR` / the CSV paths at the Kaggle dataset mounts to get a real
submission. Test studies go through the same one-time precompute step before inference.

In [45]:
test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
test_priority_series = build_priority_series(test_series_df)

precompute_features(test_df['StudyInstanceUID'].tolist(), test_priority_series, backbone, FEATURE_CACHE_DIR)

test_ds = KneeFeatureDataset(test_df['StudyInstanceUID'].tolist(), FEATURE_CACHE_DIR)
test_loader = DataLoader(test_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=TRAIN_NUM_WORKERS)

fold_preds = []
test_uids = None
for fold, ckpt_path in fold_ckpts.items():
    model = KneeFeatureModel(feat_dim=1280, n_targets=len(TARGET_COLS)).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    probs, uids = predict(model, test_loader)
    fold_preds.append(probs)
    test_uids = uids

final_probs = np.mean(fold_preds, axis=0)
sub = pd.DataFrame(final_probs, columns=TARGET_COLS)
sub.insert(0, 'StudyInstanceUID', test_uids)
sub = sub[pd.read_csv(SAMPLE_SUB_CSV).columns]   # match exact column order

sub_path = os.path.join(OUTPUT_DIR, 'submission_v1.csv')
sub.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
sub.head()

0 / 3 studies need feature extraction (11 loader threads, batch size 16)
Saved: D:/Coding/kaggle/RSNA/output/v1\submission_v1.csv


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.21236,0.282061,0.346121,0.258687,0.316793,0.332556,0.466946,0.307617,0.358326,0.356268,0.251232,0.278766
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.21236,0.282061,0.346121,0.258687,0.316793,0.332556,0.466946,0.307617,0.358326,0.356268,0.251232,0.278766
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.21236,0.282061,0.346121,0.258687,0.316793,0.332556,0.466946,0.307617,0.358326,0.356268,0.251232,0.278766


## What v2 should add

- Pool over **all** series in a plane, not just one priority series -- ~65% of study-plane
  pairs have 2+ series, this is the biggest known modeling gap in v1.
- Per-label learned attention over the 3 plane embeddings (the "region-guided" idea from
  `docs/model_architecture_plan.md`), instead of flat concatenation.
- Slice-level augmentation is currently dropped (precomputed features are deterministic).
  If it's missed, precompute two cached versions (flipped / unflipped) and sample between
  them at train time, or reintroduce augmentation once the backbone is unfrozen (next point).
- Unfreeze the last block or two of the backbone once the head has converged (Phase 2
  fine-tuning) -- at that point the feature cache goes stale and training has to switch back
  to the raw end-to-end path (`KneeMultiPlaneDataset` + `KneeMultiPlaneModel`, Section 2).
- Try swapping EfficientNet-B0 for DINOv2-Small or ConvNeXt-Tiny once the pipeline is
  validated -- `src/model/dino_backbone.py` already has a working DINOv2-via-timm wrapper to
  borrow from.
- Widen `FOLDS_TO_RUN` to all 5 and average the OOF predictions for the real CV estimate.